# LEGACY
- option to expand middle layer + validity layers etc

In [1]:
import napari
import tifffile
import numpy as np
from pathlib import Path
from magicgui.widgets import PushButton
from qtpy.QtCore import QTimer

In [2]:
combine_input_folder = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\fluorescent_cells_tifs\STUFF_FOR_BEL_ONLY\WIP_Bel_2_channel_images_with_separate_masks")
combine_output_folder = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\fluorescent_cells_tifs\3_channel_images_to_curate")
combine_output_folder.mkdir(parents=True, exist_ok=True)

# Target pixel size in microns (z, y, x)
PX_Z, PX_Y, PX_X = 5.0, 2.0, 2.0


def peek_raw(path):
    """Read a tif's array plus the raw shape/axes tifffile reports, with no
    reshaping. Used both for conversion and for error diagnostics."""
    with tifffile.TiffFile(path) as tif:
        series = tif.series[0]
        arr = np.asarray(series.asarray())
        raw_axes = series.axes.upper()
    return arr, arr.shape, raw_axes


def to_zcyx(arr, raw_axes, name):
    """Reshape a raw array + axis labels into (Z, C, Y, X).

    Singleton dimensions are squeezed away first (a 1-channel segmentation is
    often stored as (Z, 1, Y, X) or (1, Z, Y, X)), then the remaining axes are
    mapped to Z, C, Y, X.

    tifffile labels a bare multi-page stack with a generic axis ('I', 'Q', 'S',
    'T', ...) when the file carries no hyperstack metadata. Any such non-singleton
    axis that isn't already C/Y/X is treated as the Z (stacking) axis.
    """
    if len(raw_axes) != arr.ndim:
        # Axis labels are unreliable: infer purely from the squeezed shape.
        arr = np.squeeze(arr)
        raw_axes = {2: "YX", 3: "ZYX", 4: "ZCYX"}.get(arr.ndim)
        if raw_axes is None:
            raise ValueError(f"Cannot infer axes for shape {arr.shape} ({name})")

    # Normalise unknown/generic axis labels (I, Q, S, T, ...) to Z, since the
    # only non-channel stacking axis we expect is depth.
    axes = "".join(a if a in "ZCYX" else "Z" for a in raw_axes)

    # Keep Y/X always; keep other axes only if they are non-singleton.
    keep_axis = [s > 1 or axes[i] in "YX" for i, s in enumerate(arr.shape)]
    arr = arr[tuple(slice(None) if k else 0 for k in keep_axis)]
    axes = "".join(axes[i] for i, k in enumerate(keep_axis) if k)

    # Add any missing leading axes (C then Z) as size-1 so we always have ZCYX.
    for ax in ("C", "Z"):
        if ax not in axes:
            arr = np.expand_dims(arr, 0)
            axes = ax + axes

    order = [axes.index(a) for a in "ZCYX"]
    return np.transpose(arr, order)


def describe(path):
    """Best-effort raw shape/axes string for diagnostics (never raises)."""
    try:
        _, shape, axes = peek_raw(path)
        return f"raw shape={shape} axes={axes!r}"
    except Exception as exc:
        return f"<could not read: {exc}>"


image_paths = sorted(p for p in combine_input_folder.glob("*.tif") if not p.stem.endswith("_segmentation"))
print(f"{len(image_paths)} images found")

# Skip images whose combined output already exists in the output folder.
already_done = {p.name for p in combine_output_folder.glob("*.tif")}

problems = []   # (filename, reason)
skipped = 0
written = 0

for img_path in image_paths:
    out_path = combine_output_folder / img_path.name

    if img_path.name in already_done:
        skipped += 1
        continue

    seg_path = img_path.with_name(f"{img_path.stem}_segmentation.tif")

    try:
        if not seg_path.exists():
            print(f"[MISSING SEG]  {img_path.name} -> no {seg_path.name}")
            problems.append((img_path.name, "missing segmentation"))
            continue

        # Read raw arrays/axes first and print them BEFORE any reshaping, so the
        # diagnostics are always visible even if the ZCYX conversion fails.
        img_arr, img_raw_shape, img_raw_axes = peek_raw(img_path)
        seg_arr, seg_raw_shape, seg_raw_axes = peek_raw(seg_path)

        print(f"\n{img_path.name}")
        print(f"    image: raw shape={img_raw_shape} axes={img_raw_axes!r}")
        print(f"    seg:   raw shape={seg_raw_shape} axes={seg_raw_axes!r}")

        img = to_zcyx(img_arr, img_raw_axes, img_path.name)   # expect (Z, 2, Y, X)
        seg = to_zcyx(seg_arr, seg_raw_axes, seg_path.name)   # expect (Z, 1, Y, X)

        print(f"    image -> ZCYX {img.shape}")
        print(f"    seg   -> ZCYX {seg.shape}")

        if img.shape[0] != seg.shape[0]:
            print(f"    [Z MISMATCH]  image z={img.shape[0]} vs seg z={seg.shape[0]} -- skipped")
            problems.append((img_path.name, f"z mismatch {img.shape[0]} vs {seg.shape[0]}"))
            continue
        if img.shape[2:] != seg.shape[2:]:
            print(f"    [XY MISMATCH] image yx={img.shape[2:]} vs seg yx={seg.shape[2:]} -- skipped")
            problems.append((img_path.name, f"xy mismatch {img.shape[2:]} vs {seg.shape[2:]}"))
            continue
        if img.shape[1] != 2:
            print(f"    [CHANNEL?]    image has {img.shape[1]} channels (expected 2) -- skipped")
            problems.append((img_path.name, f"{img.shape[1]} channels (expected 2)"))
            continue
        if seg.shape[1] != 1:
            print(f"    [SEG CHANNEL?] segmentation has {seg.shape[1]} channels (expected 1) -- skipped")
            problems.append((img_path.name, f"seg has {seg.shape[1]} channels (expected 1)"))
            continue

        dtype = np.promote_types(img.dtype, seg.dtype)
        out = np.empty((img.shape[0], 3) + img.shape[2:], dtype=dtype)
        out[:, 0] = img[:, 0]      # channel 0: original image c0
        out[:, 1] = img[:, 1]      # channel 1: original image c1
        out[:, 2] = seg[:, 0]      # channel 2: segmentation

        tifffile.imwrite(out_path, out, imagej=True, resolution=(1.0 / PX_X, 1.0 / PX_Y),  metadata={"axes": "ZCYX", "spacing": PX_Z, "unit": "micron"})
        written += 1
        print(f"    saved -> {out_path.name}  shape={out.shape}  dtype={dtype}")

    except Exception as exc:
        # Print raw shape/axes of BOTH files so axis-ordering issues are obvious.
        print(f"[ERROR]        {img_path.name}: {exc}")
        print(f"    image: {describe(img_path)}")
        print(f"    seg:   {describe(seg_path)}")
        problems.append((img_path.name, f"error: {exc}"))
        continue

print(f"\nDone. {written} written, {skipped} already-done (skipped), "
      f"{len(problems)} flagged.")
if problems:
    print("Flagged files:")
    for name, reason in problems:
        print(f"  - {name}: {reason}")

12 images found

20260611_Exp6_Vacumap_BF-RFP_Ibuprofen_M1-4_Merged.tif
    image: raw shape=(26, 2, 3785, 6503) axes='ZCYX'
    seg:   raw shape=(26, 3785, 6503) axes='IYX'
    image -> ZCYX (26, 2, 3785, 6503)
    seg   -> ZCYX (26, 1, 3785, 6503)
    saved -> 20260611_Exp6_Vacumap_BF-RFP_Ibuprofen_M1-4_Merged.tif  shape=(26, 3, 3785, 6503)  dtype=uint8

20260616_Exp9_Vascumap_BF_RFP_Thunder_DMSO_Paclitaxel_M2-3_DMSO.tif
    image: raw shape=(42, 2, 6523, 3773) axes='ZCYX'
    seg:   raw shape=(42, 6523, 3773) axes='ZYX'
    image -> ZCYX (42, 2, 6523, 3773)
    seg   -> ZCYX (42, 1, 6523, 3773)
    saved -> 20260616_Exp9_Vascumap_BF_RFP_Thunder_DMSO_Paclitaxel_M2-3_DMSO.tif  shape=(42, 3, 6523, 3773)  dtype=uint8

CART_day7_FL32_FL32_ARi2_Merged.tif
    image: raw shape=(26, 2, 2114, 4744) axes='ZCYX'
    seg:   raw shape=(26, 2114, 4744) axes='ZYX'
    image -> ZCYX (26, 2, 2114, 4744)
    seg   -> ZCYX (26, 1, 2114, 4744)
    saved -> CART_day7_FL32_FL32_ARi2_Merged.tif  shape=(26

c:\Users\taylorhearn\AppData\Local\miniforge3\envs\nap-ij\Lib\site-packages\tifffile\tifffile.py:3711: UserWarning: <tifffile.TiffWriter 'day5_6mm_dish1_…vice7_Merged.tif'> truncating ImageJ file
  self._write_remaining_pages()


    saved -> day5_6mm_dish1_device7_Merged_day5_6mm_dish1_device7_Merged.tif  shape=(46, 3, 7800, 5172)  dtype=uint16

day5_device7_FAIL_Device7_Merged.tif
    image: raw shape=(36, 2, 4086, 3156) axes='ZCYX'
    seg:   raw shape=(36, 4086, 3156) axes='IYX'
    image -> ZCYX (36, 2, 4086, 3156)
    seg   -> ZCYX (36, 1, 4086, 3156)
    saved -> day5_device7_FAIL_Device7_Merged.tif  shape=(36, 3, 4086, 3156)  dtype=uint16

day7_device11_Device11_Merged.tif
    image: raw shape=(66, 2, 2857, 4688) axes='ZCYX'
    seg:   raw shape=(66, 2857, 4688) axes='ZYX'
    image -> ZCYX (66, 2, 2857, 4688)
    seg   -> ZCYX (66, 1, 2857, 4688)


c:\Users\taylorhearn\AppData\Local\miniforge3\envs\nap-ij\Lib\site-packages\tifffile\tifffile.py:3711: UserWarning: <tifffile.TiffWriter 'day7_device11_D…ice11_Merged.tif'> truncating ImageJ file
  self._write_remaining_pages()


    saved -> day7_device11_Device11_Merged.tif  shape=(66, 3, 2857, 4688)  dtype=uint16

Farid_Thunder_170326_Reservoirs_day7_Reservoirs_Reservoir_1_Merged.tif
    image: raw shape=(48, 2, 6505, 2859) axes='ZCYX'
    seg:   raw shape=(48, 6505, 2859) axes='ZYX'
    image -> ZCYX (48, 2, 6505, 2859)
    seg   -> ZCYX (48, 1, 6505, 2859)


c:\Users\taylorhearn\AppData\Local\miniforge3\envs\nap-ij\Lib\site-packages\tifffile\tifffile.py:3711: UserWarning: <tifffile.TiffWriter 'Farid_Thunder_17…ir_1_Merged.tif'> truncating ImageJ file
  self._write_remaining_pages()


    saved -> Farid_Thunder_170326_Reservoirs_day7_Reservoirs_Reservoir_1_Merged.tif  shape=(48, 3, 6505, 2859)  dtype=uint16

Farid_Thunder_170326_Reservoirs_day7_Reservoirs_Reservoir_3_Merged.tif
    image: raw shape=(60, 2, 6524, 2857) axes='ZCYX'
    seg:   raw shape=(60, 6524, 2857) axes='IYX'
    image -> ZCYX (60, 2, 6524, 2857)
    seg   -> ZCYX (60, 1, 6524, 2857)


c:\Users\taylorhearn\AppData\Local\miniforge3\envs\nap-ij\Lib\site-packages\tifffile\tifffile.py:3711: UserWarning: <tifffile.TiffWriter 'Farid_Thunder_17…ir_3_Merged.tif'> truncating ImageJ file
  self._write_remaining_pages()


    saved -> Farid_Thunder_170326_Reservoirs_day7_Reservoirs_Reservoir_3_Merged.tif  shape=(60, 3, 6524, 2857)  dtype=uint16

GFP_HUV_ML_From_Luca_Tina_Stellaris_G014_5_1R_Merged.tif
    image: raw shape=(90, 2, 2184, 1701) axes='ZCYX'
    seg:   raw shape=(90, 2184, 1701) axes='IYX'
    image -> ZCYX (90, 2, 2184, 1701)
    seg   -> ZCYX (90, 1, 2184, 1701)
    saved -> GFP_HUV_ML_From_Luca_Tina_Stellaris_G014_5_1R_Merged.tif  shape=(90, 3, 2184, 1701)  dtype=uint8

GFP_Lung_Fibros_From_Luca_Tina_Stellaris_11_Merged.tif
    image: raw shape=(64, 2, 2354, 1443) axes='ZCYX'
    seg:   raw shape=(64, 2354, 1443) axes='IYX'
    image -> ZCYX (64, 2, 2354, 1443)
    seg   -> ZCYX (64, 1, 2354, 1443)
    saved -> GFP_Lung_Fibros_From_Luca_Tina_Stellaris_11_Merged.tif  shape=(64, 3, 2354, 1443)  dtype=uint8

Done. 11 written, 1 already-done (skipped), 0 flagged.
